# Intrusion Detection System (IDS)

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install pandas scikit-learn numpy matplotlib seaborn
!pip install scapy



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Try importing scapy, fallback if not installed
try:
    from scapy.all import sniff
    SCAPY_AVAILABLE = True
except ImportError:
    print("[WARNING] Scapy not installed. Real-time sniffing disabled.")
    SCAPY_AVAILABLE = False

# Try to load real dataset, else use mock data and the data is raw
try:
    df = pd.read_csv("CICIDS2017.csv")
    print("[INFO] Loaded CICIDS2017.csv")
except FileNotFoundError:
    print("[WARNING] CICIDS2017.csv not found. Using mock dataset.")
    data = {
        'feature1': [1, 2, 1.5, 8, 9, 7.5],
        'feature2': [1, 1.2, 0.8, 9, 8.5, 9.1],
        'Label':    [0, 0, 0, 1, 1, 1]
    }
    df = pd.DataFrame(data)

# Encode labels (Normal = 0, Attack = 1)
if 'Label' in df.columns and df['Label'].dtype != 'int':
    df['Label'] = LabelEncoder().fit_transform(df['Label'])

# Drop non-numeric columns (if any)
df = df.select_dtypes(include=['number'])

# Handle missing values
df = df.fillna(df.median())

# Add a timestamp column if missing
df['ts'] = time.time()

# Split into features (X) and target labels (y)
X = df.drop(columns=["Label"])
y = df["Label"]

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Test the model
y_pred = clf.predict(X_test)

# Evaluate performance
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Optional: Real-time packet monitoring
if SCAPY_AVAILABLE:
    def packet_callback(packet):
        features = np.random.rand(1, X_train.shape[1])  # Simulated features
        prediction = clf.predict(features)
        if prediction == 1:
            print(f"[ALERT] Potential intrusion detected at {time.strftime('%Y-%m-%d %H:%M:%S')}!")

    print("\n[INFO] Starting real-time intrusion detection... Press CTRL+C to stop.")
    try:
        sniff(filter="tcp", prn=packet_callback, store=False)
    except Exception as e:
        print(f"[WARNING] Real-time detection disabled due to error: {e}")
else:
    print("[INFO] Real-time detection not available without scapy.")